# Domain 5 — Model Selection and Optimization (16.8%)

| Skill | Weight |
|---|---|
| Technical Fundamentals | 6.1% |
| LLM Fundamentals | 5.2% |
| Cost and Token Management | 2.8% |
| Model Selection and Tradeoffs | 2.7% |


In [ ]:
"""Shared setup. Export ANTHROPIC_API_KEY before launching Jupyter."""

import json
import os

import anthropic

client = anthropic.Anthropic()

# Current self-serve model IDs (verified against Anthropic docs, Sept 2026).
OPUS = "claude-opus-5"
SONNET = "claude-sonnet-5"
HAIKU = "claude-haiku-4-5-20251001"

MODEL = SONNET


def extract_text(response: anthropic.types.Message) -> str:
    """Concatenate text blocks, ignoring thinking and tool_use blocks."""
    return "".join(
        block.text for block in response.content if block.type == "text"
    )


print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


## 5.1 LLM Fundamentals — counting tokens before you spend them

The token-counting endpoint tells you the input size of a request **without
running it**. This is how you enforce a budget at the boundary instead of
discovering the cost on the invoice.

Note that tokenisers differ between model generations, so a count is only
valid for the model you asked about.


In [ ]:
def count_input_tokens(model: str, system: str, user_text: str) -> int:
    """Return the input token count for a request, without sending it."""
    result = client.messages.count_tokens(
        model=model,
        system=system,
        messages=[{"role": "user", "content": user_text}],
    )
    return result.input_tokens


system_prompt = "You are a claims triage classifier."
claim = "Hail damage to roof; adjuster estimate $8,400; policy covers hail."

for model in (SONNET, HAIKU):
    tokens = count_input_tokens(model, system_prompt, claim)
    print(f"{model:32} {tokens:>5} input tokens")


In [ ]:
def enforce_token_budget(
    model: str, system: str, user_text: str, budget: int
) -> None:
    """Reject a request that would exceed the input token budget.

    Checking before the call is the point: once you send it you have
    already paid for it.
    """
    tokens = count_input_tokens(model, system, user_text)
    if tokens > budget:
        raise ValueError(
            f"request needs {tokens} input tokens, budget is {budget}"
        )
    print(f"within budget: {tokens}/{budget} tokens")


enforce_token_budget(MODEL, system_prompt, claim, budget=100)

try:
    enforce_token_budget(MODEL, system_prompt, "word " * 500, budget=100)
except ValueError as error:
    print("rejected:", error)


### Extended thinking

Extended thinking gives the model a scratchpad before it answers. You buy
accuracy on multi-step reasoning with tokens and latency.

Mechanics worth knowing: `budget_tokens` must be **less than** `max_tokens`,
the thinking appears as a `thinking` content block before the `text` block,
and thinking tokens are billed as output.

The exam angle is the tradeoff, not the syntax: enabling thinking on a
simple classification is wasted money; disabling it on a multi-constraint
reasoning problem costs you correctness.


In [ ]:
PUZZLE = (
    "A claim was filed 45 days after the loss. The policy requires filing "
    "within 30 days, except that a declared state of emergency extends the "
    "deadline by 60 days. The emergency was declared 10 days BEFORE the "
    "loss and lifted 5 days after it. Is the filing timely? Answer yes or "
    "no, then give the deadline date arithmetic."
)


def answer_without_thinking(question: str) -> anthropic.types.Message:
    """Standard call: the model answers directly."""
    return client.messages.create(
        model=MODEL,
        max_tokens=600,
        messages=[{"role": "user", "content": question}],
    )


def answer_with_thinking(
    question: str, budget_tokens: int = 2000
) -> anthropic.types.Message:
    """Extended thinking call. budget_tokens must be < max_tokens."""
    return client.messages.create(
        model=MODEL,
        max_tokens=budget_tokens + 1000,
        thinking={"type": "enabled", "budget_tokens": budget_tokens},
        messages=[{"role": "user", "content": question}],
    )


plain = answer_without_thinking(PUZZLE)
print("WITHOUT THINKING")
print("blocks:", [block.type for block in plain.content])
print("output tokens:", plain.usage.output_tokens)
print(extract_text(plain))

deep = answer_with_thinking(PUZZLE)
print("\n\nWITH THINKING")
print("blocks:", [block.type for block in deep.content])
print("output tokens:", deep.usage.output_tokens)
print(extract_text(deep))


### Non-determinism

The same prompt does not guarantee the same output. `temperature=0` makes
sampling greedy and *much* more consistent, but it is not a contractual
guarantee of byte-identical output.

The practical consequence, and the thing the exam cares about: you cannot
write a test that asserts exact string equality on model output. Assert on
structure and on properties instead.


In [ ]:
def sample_repeatedly(prompt: str, temperature: float, runs: int = 3) -> None:
    """Run the same prompt several times and show the spread."""
    outputs = []
    for _ in range(runs):
        response = client.messages.create(
            model=MODEL,
            max_tokens=60,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        outputs.append(extract_text(response).strip())

    print(f"temperature={temperature}")
    for index, text in enumerate(outputs, start=1):
        print(f"  run {index}: {text}")
    print(f"  distinct outputs: {len(set(outputs))}/{runs}\n")


prompt = "Invent a one-line tagline for a claims processing tool."
sample_repeatedly(prompt, temperature=1.0)
sample_repeatedly(prompt, temperature=0.0)


## 5.2 Cost and Token Management (2.8%)

Rates below are from Anthropic's pricing docs as of September 2026. Prices
change — the exam tests the *shape* of the calculation and the relative
ordering of the tiers, not the digits.

Two multipliers matter most:

- **Batch API**: 50% off input and output.
- **Prompt caching**: a cache read costs 0.1x base input. A 5-minute cache
  write costs 1.25x, a 1-hour write costs 2x. So a 5-minute cache pays for
  itself after **one** read; a 1-hour cache after **two**.


In [ ]:
# USD per million tokens, from the Anthropic pricing docs (Sept 2026).
RATES = {
    "claude-opus-5": {"input": 5.00, "output": 25.00},
    "claude-sonnet-5": {"input": 2.00, "output": 10.00},
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

CACHE_READ_MULTIPLIER = 0.1
CACHE_WRITE_5M_MULTIPLIER = 1.25
CACHE_WRITE_1H_MULTIPLIER = 2.0
BATCH_MULTIPLIER = 0.5


def estimate_cost(
    model: str,
    input_tokens: int = 0,
    output_tokens: int = 0,
    cache_read_tokens: int = 0,
    cache_write_tokens: int = 0,
    batch: bool = False,
    cache_ttl: str = "5m",
) -> float:
    """Return the USD cost of a request.

    Cached reads are billed at a fraction of base input; cache writes at a
    premium. The batch discount multiplies the whole total.
    """
    rate = RATES[model]
    write_multiplier = (
        CACHE_WRITE_5M_MULTIPLIER
        if cache_ttl == "5m"
        else CACHE_WRITE_1H_MULTIPLIER
    )

    cost = (
        input_tokens * rate["input"]
        + output_tokens * rate["output"]
        + cache_read_tokens * rate["input"] * CACHE_READ_MULTIPLIER
        + cache_write_tokens * rate["input"] * write_multiplier
    ) / 1_000_000

    return cost * BATCH_MULTIPLIER if batch else cost


# 10,000 documents, 2,000 input / 500 output tokens each.
volume, tokens_in, tokens_out = 10_000, 2_000, 500

for model in RATES:
    realtime = volume * estimate_cost(model, tokens_in, tokens_out)
    batched = volume * estimate_cost(model, tokens_in, tokens_out, batch=True)
    print(
        f"{model:32} realtime ${realtime:8.2f}   batch ${batched:8.2f}"
    )


In [ ]:
def cache_breakeven(model: str, cached_tokens: int, ttl: str) -> None:
    """Show after how many reads a cache write pays for itself."""
    rate = RATES[model]["input"]
    multiplier = (
        CACHE_WRITE_5M_MULTIPLIER if ttl == "5m" else CACHE_WRITE_1H_MULTIPLIER
    )

    uncached_per_call = cached_tokens * rate / 1_000_000
    write_once = cached_tokens * rate * multiplier / 1_000_000
    read_per_call = cached_tokens * rate * CACHE_READ_MULTIPLIER / 1_000_000

    print(f"{model} | {cached_tokens:,} cached tokens | ttl={ttl}")
    print(f"  no cache, per call : ${uncached_per_call:.4f}")
    print(f"  cache write (once) : ${write_once:.4f}")
    print(f"  cache read per call: ${read_per_call:.4f}")

    for calls in range(1, 6):
        without = uncached_per_call * calls
        with_cache = write_once + read_per_call * (calls - 1)
        verdict = "cache wins" if with_cache < without else "cache loses"
        print(
            f"  {calls} call(s): ${without:.4f} vs ${with_cache:.4f} "
            f"-> {verdict}"
        )
    print()


cache_breakeven(MODEL, cached_tokens=50_000, ttl="5m")
cache_breakeven(MODEL, cached_tokens=50_000, ttl="1h")


In [ ]:
# A long, stable prefix is the thing worth caching: a policy manual, a
# style guide, a schema. Mark it with cache_control and keep it BYTE
# IDENTICAL across calls -- any change invalidates the cache.
POLICY_MANUAL = (
    "SECTION 1. COVERED PERILS. Fire, lightning, windstorm, hail, "
    "explosion, riot, aircraft, vehicles, smoke, vandalism, theft, "
    "falling objects, weight of ice and snow, accidental water discharge. "
) * 120


def ask_with_cache(question: str) -> anthropic.types.Message:
    """Send a question against a cached policy manual prefix.

    cache_control marks the breakpoint: everything up to and including
    this block is cacheable. Variable content must come after it.
    """
    return client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": "You answer coverage questions from the manual.",
            },
            {
                "type": "text",
                "text": POLICY_MANUAL,
                "cache_control": {"type": "ephemeral"},
            },
        ],
        messages=[{"role": "user", "content": question}],
    )


def show_cache_usage(label: str, response: anthropic.types.Message) -> None:
    """Print the cache-specific fields of a usage object."""
    usage = response.usage
    print(
        f"{label:12} input={usage.input_tokens:<6} "
        f"cache_write={usage.cache_creation_input_tokens:<6} "
        f"cache_read={usage.cache_read_input_tokens:<6}"
    )


show_cache_usage("first call", ask_with_cache("Is hail covered?"))
show_cache_usage("second call", ask_with_cache("Is flood covered?"))
show_cache_usage("third call", ask_with_cache("Is theft covered?"))


## 5.3 Model Selection and Tradeoffs (2.7%)

The tiers, in plain terms:

- **Haiku** — fastest and cheapest. Classification, extraction, routing,
  high-volume simple work.
- **Sonnet** — the default for most production workloads. Balanced.
- **Opus** — complex agentic work, long multi-step reasoning, hard code.

The exam gives you a scenario with constraints and asks which tier fits.
The trap answer is always "use the biggest model" — a scenario that
specifies high volume and simple work is testing whether you downshift.


In [ ]:
def recommend_model(
    complexity: str, latency_critical: bool, volume_per_day: int
) -> str:
    """Recommend a tier from task shape.

    complexity: "simple" | "moderate" | "complex".
    """
    if complexity == "complex":
        return f"{OPUS} (capability dominates; accept cost and latency)"

    if complexity == "simple" and (latency_critical or volume_per_day > 5_000):
        return f"{HAIKU} (simple work at speed/scale; cheapest per call)"

    if latency_critical and complexity == "moderate":
        return (
            f"{HAIKU} (latency budget rules out larger tiers; "
            "verify quality with evals)"
        )

    return f"{SONNET} (balanced default for production workloads)"


scenarios = [
    ("Route 50k support emails to a queue", "simple", True, 50_000),
    ("Draft a nuanced denial letter", "moderate", False, 300),
    ("Multi-step agent refactoring a codebase", "complex", False, 50),
    ("Autocomplete in an IDE", "simple", True, 200_000),
]

for label, complexity, latency, volume in scenarios:
    print(f"{label}\n  -> {recommend_model(complexity, latency, volume)}\n")


In [ ]:
def compare_tiers(prompt: str, max_tokens: int = 200) -> None:
    """Run one prompt across all three tiers and report cost and latency.

    This is how you should actually pick a model: measure on YOUR task,
    not on a benchmark someone else ran.
    """
    import time

    for model in (HAIKU, SONNET, OPUS):
        start = time.perf_counter()
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        elapsed = time.perf_counter() - start
        cost = estimate_cost(
            model,
            response.usage.input_tokens,
            response.usage.output_tokens,
        )
        print(f"{model:32} {elapsed:5.2f}s  ${cost:.6f}")
        print(f"  {extract_text(response).strip()[:110]}\n")


compare_tiers("Classify this claim in one word: roof hail damage, covered.")


## 5.4 Technical Fundamentals (6.1%)

The SDK is a thin convenience layer over a plain REST endpoint. Seeing the
raw call once means you will not be confused by an exam item that describes
HTTP-level behaviour — headers, status codes, the JSON body shape.


In [ ]:
import requests

API_URL = "https://api.anthropic.com/v1/messages"


def raw_rest_call(prompt: str) -> dict:
    """Call the Messages API over plain HTTP, no SDK.

    Three headers are mandatory: the API key, the API version, and the
    content type. The response JSON has the same shape the SDK parses
    into objects.
    """
    response = requests.post(
        API_URL,
        headers={
            "x-api-key": os.environ["ANTHROPIC_API_KEY"],
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": MODEL,
            "max_tokens": 100,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


payload = raw_rest_call("Reply with exactly: raw REST works.")
print(json.dumps(payload, indent=2)[:700])
print("\nSame shape the SDK wraps: content is a list of typed blocks.")
